In [2]:
import sys

sys.argv = [
    "ipykernel_launcher.py",  
    "--dataset", "CustomAWGN30ES15",  # Ensure value is provided
    "--model", "",
    "--Device", "cpu",
    "--test"  # This is a flag, so no value needed
]

In [3]:
import pickle
import os
import pandas as pd
import numpy as np
import sqlite3
from tqdm import tqdm
import copy

from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn
from scipy.signal import resample
from src.models import *
from src.utils import *
from main import  load_dataset, backprop

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.dates import DateFormatter

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)


In [10]:
feature_set = ['Active Power', 'Reactive Power', 'Governor speed actual', 'UGB X displacement', 'UGB Y displacement',
    'LGB X displacement', 'LGB Y displacement', 'TGB X displacement',
    'TGB Y displacement', 'Stator winding temperature 13',
    'Stator winding temperature 14', 'Stator winding temperature 15',
    'Surface Air Cooler Air Outlet Temperature',
    'Surface Air Cooler Water Inlet Temperature',
    'Surface Air Cooler Water Outlet Temperature',
    'Stator core temperature', 'UGB metal temperature',
    'LGB metal temperature 1', 'LGB metal temperature 2',
    'LGB oil temperature', 'Penstock Flow', 'Turbine flow',
    'UGB cooling water flow', 'LGB cooling water flow',
    'Generator cooling water flow', 'Governor Penstock Pressure',
    'Penstock pressure', 'Opening Wicked Gate', 'UGB Oil Contaminant',
    'Gen Thrust Bearing Oil Contaminant']

feature_tag_mapping = {
    'Stator winding temperature 13': 'U-LGS1-TI-81104A-AI',
    'Stator winding temperature 14': 'U-LGS1-TI-81104B-AI',
    'Stator winding temperature 15': 'U-LGS1-TI-81104C-AI',
    'Surface Air Cooler Air Outlet Temperature': 'U-LGS1-TI-81104D-AI',
    'Surface Air Cooler Water Inlet Temperature': 'U-LGS1-TI-81104E-AI',
    'Surface Air Cooler Water Outlet Temperature': 'U-LGS1-TI-81104F-AI',
    'Stator core temperature': 'U-LGS1-TI-81104G-AI',
    'UGB metal temperature': 'U-LGS1-TI-81104H-AI',
    'UGB oil temperature': 'U-LGS1-TI-81104I-AI',
    'LGB metal temperature 1': 'U-LGS1-TI-81104J-AI',
    'LGB metal temperature 2': 'U-LGS1-TI-81104K-AI',
    'LGB oil temperature': 'U-LGS1-TI-81104L-AI',
    'Governor speed actual': 'U-LGS1-SI-81101-AI',
    'UGB X displacement': 'U-LGS1-UGB-X-PK-PK-70-AI',
    'UGB Y displacement': 'U-LGS1-UGB-Y-PK-PK-340-AI',
    'LGB X displacement': 'U-LGS1-GB-X-PK-PK-70-AI',
    'LGB Y displacement': 'U-LGS1-LGB-Y-PK-PK-340-AI',
    'TGB X displacement': 'U-LGS1-TGB-X-PK-PK-270-AI',
    'TGB Y displacement': 'U-LGS1-TGB-Y-PK-PK-340-AI',
    'Active Power': 'U-LGS1-Active-Power-AI',
    'Reactive Power': 'U-LGS1-Reactive-Power-AI',
    'Grid Selection': 'U-LGS1-N75-15-0-AI',
    'Opening Wicked Gate': 'U-LGS1-ZT-81101-AI',
    'UGB Oil Contaminant': 'U-LGS1-AY-81103B-AI',
    'Gen Thrust Bearing Oil Contaminant': 'U-LGS1-AY-81103C-AI',
    'Gen Voltage Phase 1': 'U-LGS1-EI_81151A_MV-AI',
    'Gen Voltage Phase 2': 'U-LGS1-EI_81151B_MV-AI',
    'Gen Voltage Phase 3': 'U-LGS1-EI_81151C_MV-AI',
    'Gen Current Phase 1': 'U-LGS1-II_81152A_MV-AI',
    'Gen Current Phase 2': 'U-LGS1-II_81152B_MV-AI',
    'Gen Current Phase 3': 'U-LGS1-II_81152C_MV-AI',
    'Penstock Flow': 'U-LGS1-FI-81101-AI',
    'Turbine flow': 'U-LGS1-FIT-431-AI',
    'UGB cooling water flow': 'U-LGS1-FIT-81103A-AI',
    'LGB cooling water flow': 'U-LGS1-FIT-81103B-AI',
    'Generator cooling water flow': 'U-LGS1-FIT-81103C-AI',
    'Governor Penstock Pressure': 'U-LGS1-PI-81101-AI',
    'Penstock pressure': 'U-LGS1-PT-81150-AI'
}

pi_tag = [feature_tag_mapping[feature] for feature in feature_set + ['Grid Selection']]

In [24]:
master_pd = ""
for i in range(len(pi_tag)):
    value_resp = pd.read_csv(f'data/DataPI2025/{pi_tag[i]}.csv')
    if i == 0:
        value_resp['Timestamps'] = pd.to_datetime(value_resp['Timestamps'])
        master_pd = value_resp
    else:
        master_pd = pd.concat([master_pd, value_resp['Values']], axis=1, join='inner')

master_pd = master_pd.values
master_pd = pd.DataFrame(data=master_pd, columns=['TimeStamp'] + feature_set + ['Grid Selection'])
master_pd = master_pd.reset_index(drop=True)
master_pd.replace('I/O Timeout', np.nan, inplace=True)

for column_name in master_pd.columns:
    if column_name != 'Load_Type' and column_name != 'TimeStamp':
        master_pd[column_name] = pd.to_numeric(master_pd[column_name], downcast='float')

In [27]:
def fill_nans_conditionally(series, threshold=100):
    is_nan = series.isna()
    groups = (is_nan != is_nan.shift()).cumsum()
    
    nan_counts = is_nan.groupby(groups).transform('sum')
    series_filled = series.copy()
    series_filled[nan_counts < threshold] = series_filled[nan_counts < threshold].fillna(method='ffill')
    return series_filled

def consecutive_nan_info(df, col, timestamp_col='TimeStamp'):
    is_nan = df[col].isna()
    groups = (is_nan != is_nan.shift()).cumsum()
    
    results = []
    for group_id, group_data in df.groupby(groups):
        if group_data[col].isna().all():
            count = group_data.shape[0]
            start_time = group_data[timestamp_col].iloc[0]
            end_time = group_data[timestamp_col].iloc[-1]
            results.append({"start_time": start_time, "end_time": end_time, "nan_count": count})
    return results

In [31]:
sensor_columns = [col for col in master_pd.columns if col not in ["TimeStamp"]]
nan_info_all = {}

for col in sensor_columns:
    nan_info_all[col] = consecutive_nan_info(master_pd, col)

for sensor, info in nan_info_all.items():
    for group in info:
        print(group)

In [30]:
sensor_columns = [col for col in master_pd.columns if col != "TimeStamp"]
master_pd[sensor_columns] = master_pd[sensor_columns].apply(lambda col: fill_nans_conditionally(col))

In [34]:
master_pd = master_pd.sort_values(by='TimeStamp')
master_pd.reset_index(drop=True, inplace=True)

In [36]:
master_pd.to_csv("PI2025-Now.csv", index=False)